# Wikidata Candidate Sense Merge Experiment

This notebook tests whether an OpenAI API LLM can merge Wikidata candidate senses that are semantically close or duplicated. It uses the same candidate-loading utility used by `LiteSemRAG` semantic-description assignment, then asks the LLM to produce a smaller cleaned candidate set.

## Parameters

Change `WORD` and `MAX_CANDIDATE_COUNT`, then run the notebook top to bottom. The OpenAI key is read from `OPENAI_API_KEY` if present, otherwise from the root `API_KEY` file.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import build_wikidata_candidate_bank, load_wikidata_definition_candidates

MAX_CANDIDATE_COUNT = 8
USE_DETAILED_DESCRIPTION = False

OPENAI_MODEL = "gpt-5.4-mini"
API_KEY_PATH = REPO_ROOT / "API_KEY"


def load_local_api_config(path: Path = API_KEY_PATH) -> dict:
    if not path.exists():
        return {}
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return {}
    if text.startswith("{"):
        return json.loads(text)
    if "=" not in text:
        return {"OPENAI_API_KEY": clean_api_value(text)}
    config = {}
    for part in text.replace(";", "\n").splitlines():
        part = part.strip()
        if not part or part.startswith("#") or "=" not in part:
            continue
        key, value = part.split("=", 1)
        config[key.strip().strip("'\"")] = clean_api_value(value)
    return config


def clean_api_value(value: str) -> str:
    value = str(value).strip()
    value = value.removesuffix(",").strip()
    value = value.strip("'\"")
    value = value.replace("\\n", "").replace("\\r", "").strip()
    value = value.removesuffix(",").strip()
    value = value.strip("'\"")
    return value


def first_config_value(config: dict, *keys: str):
    for key in keys:
        value = config.get(key)
        if value:
            return value
    return None


LOCAL_API_CONFIG = load_local_api_config()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or first_config_value(
    LOCAL_API_CONFIG,
    "OPENAI_API_KEY",
    "openai_api_key",
    "api_key",
    "chatgpt_api",
)
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL") or first_config_value(
    LOCAL_API_CONFIG,
    "OPENAI_BASE_URL",
    "openai_base_url",
    "base_url",
)

print(f"repo: {REPO_ROOT}")
print(f"local API key file exists: {API_KEY_PATH.exists()}")
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)}")
print(f"OpenAI base URL: {OPENAI_BASE_URL or '(default)'}")

repo: /home/xiaoyue/LiteSemRAG
local API key file exists: True
OpenAI key loaded: True
OpenAI base URL: (default)


## 1. Fetch Wikidata Candidate Senses

`MAX_CANDIDATE_COUNT` is passed both as the Wikidata search limit and the post-filter target count, so this cell requests up to that many usable definitions.

In [2]:
def fetch_candidate_senses(word: str, max_candidates: int, use_detailed_description: bool = False):
    candidates_df, definition_column = load_wikidata_definition_candidates(
        word,
        use_detailed_description=use_detailed_description,
        limit=max_candidates,
        target_candidate_count=max_candidates,
    )
    candidate_bank = build_wikidata_candidate_bank(
        candidates_df,
        definition_column=definition_column,
    )
    rows = []
    for idx, candidate in enumerate(candidate_bank, start=1):
        rows.append(
            {
                "candidate_id": idx,
                "wikidata_id": candidate["entity_id"],
                "label": candidate["label"],
                "description": candidate["description"],
                "definition": candidate["definition"],
                "hypothesis": candidate["hypothesis"],
            }
        )
    return rows, candidates_df

WORD = "season"
candidate_senses, raw_candidates_df = fetch_candidate_senses(
    WORD,
    MAX_CANDIDATE_COUNT,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
)

candidate_df = pd.DataFrame(candidate_senses)
candidate_df

,candidate_id,wikidata_id,label,description,definition,hypothesis
0,1,Q24384,season,weather- or climate-based subdivision of the year,weather- or climate-based subdivision of the year,It refers to weather- or climate-based subdivi...
1,2,Q10688145,season,"section of a year, in cultural forms (tourism,...","section of a year, in cultural forms (tourism,...","It refers to section of a year, in cultural fo..."
2,3,Q27020041,sports season,"season of a sports league or competition, gene...","season of a sports league or competition, gene...",It refers to season of a sports league or comp...
3,4,Q3464665,television series season,set of episodes produced for a television series,set of episodes produced for a television series,It refers to set of episodes produced for a te...


## 2. Prompt The LLM To Merge Similar Senses

The prompt asks the model to preserve genuinely distinct meanings, merge near-duplicates, and return strict JSON so the result can be reused by indexing experiments.

In [3]:
SYSTEM_PROMPT = """You are a careful lexical semantics annotator.
Your task is to clean candidate word senses from Wikidata for a semantic retrieval system.
Merge candidates when they refer to the same or nearly the same meaning, even if their labels or wording differ.
Keep candidates separate when they would lead to different retrieval behavior in context.
Prefer concise, concrete sense descriptions.
Return only valid JSON."""


def compact_candidates_for_prompt(candidates: list[dict]) -> list[dict]:
    return [
        {
            "candidate_id": candidate["candidate_id"],
            "label": candidate["label"],
            "hypothesis": candidate["hypothesis"],
        }
        for candidate in candidates
    ]


def build_merge_prompt(word: str, candidates: list[dict]) -> str:
    payload = {
        "word": word,
        "candidate_senses": compact_candidates_for_prompt(candidates),
    }
    return f"""Merge near-duplicate candidate senses for the target word.

Rules:
1. Merge candidates only when their hypotheses mean the same or nearly the same thing for retrieval.
2. Keep candidates separate when they describe meaningfully different contextual senses.
3. Discard candidates that are too vague, redundant, or not a plausible sense of the target word.
4. Output only valid JSON with keys: word, merged_senses, discarded_candidates, notes.

Each merged_senses item must contain:
- sense_id: a short stable id such as s1, s2, s3
- canonical_label: short label for the merged sense
- merged_description: one sentence describing the merged meaning
- source_candidate_ids: list of integer candidate_id values that were merged
- merge_rationale: one short sentence

Each discarded_candidates item must contain:
- candidate_id
- reason

Candidate data:
{json.dumps(payload, ensure_ascii=False, indent=2)}"""


merge_prompt = build_merge_prompt(WORD, candidate_senses)
print(merge_prompt)

Merge near-duplicate candidate senses for the target word.

Rules:
1. Merge candidates only when their hypotheses mean the same or nearly the same thing for retrieval.
2. Keep candidates separate when they describe meaningfully different contextual senses.
3. Discard candidates that are too vague, redundant, or not a plausible sense of the target word.
4. Output only valid JSON with keys: word, merged_senses, discarded_candidates, notes.

Each merged_senses item must contain:
- sense_id: a short stable id such as s1, s2, s3
- canonical_label: short label for the merged sense
- merged_description: one sentence describing the merged meaning
- source_candidate_ids: list of integer candidate_id values that were merged
- merge_rationale: one short sentence

Each discarded_candidates item must contain:
- candidate_id
- reason

Candidate data:
{
  "word": "season",
  "candidate_senses": [
    {
      "candidate_id": 1,
      "label": "season",
      "hypothesis": "It refers to weather- or cli

In [4]:
from openai import OpenAI


def make_openai_client():
    client_kwargs = {}
    if OPENAI_API_KEY:
        client_kwargs["api_key"] = OPENAI_API_KEY
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    return OpenAI(**client_kwargs)


def merge_candidate_senses_with_llm(
    word: str,
    candidates: list[dict],
    model: str = OPENAI_MODEL,
):
    if not OPENAI_API_KEY:
        raise RuntimeError("OpenAI API key was not found in OPENAI_API_KEY or the root API_KEY file.")

    client = make_openai_client()
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_merge_prompt(word, candidates)},
        ],
    )
    content = response.choices[0].message.content
    return json.loads(content), response


llm_result, raw_response = merge_candidate_senses_with_llm(WORD, candidate_senses)
usage = raw_response.usage
token_usage = {
    "prompt_tokens": getattr(usage, "prompt_tokens", None),
    "completion_tokens": getattr(usage, "completion_tokens", None),
    "total_tokens": getattr(usage, "total_tokens", None),
}
print(token_usage)
llm_result

{'prompt_tokens': 451, 'completion_tokens': 298, 'total_tokens': 749}


{'word': 'season',
 'merged_senses': [{'sense_id': 's1',
   'canonical_label': 'time of year',
   'merged_description': 'A recurring subdivision of the year marked by climate, weather, or other calendar-based cycles.',
   'source_candidate_ids': [1, 2],
   'merge_rationale': 'Both candidates describe a general year subdivision sense of season, with candidate 2 adding common domain-specific uses that still fit the same broad meaning.'},
  {'sense_id': 's2',
   'canonical_label': 'sports season',
   'merged_description': 'The scheduled period during which a sports league or competition takes place.',
   'source_candidate_ids': [3],
   'merge_rationale': 'This is a distinct sports-specific sense with different retrieval behavior from the general year subdivision sense.'},
  {'sense_id': 's3',
   'canonical_label': 'TV season',
   'merged_description': 'A set of episodes produced as one installment of a television series.',
   'source_candidate_ids': [4],
   'merge_rationale': 'This is a d

## 3. Inspect The Merged Candidate Set

In [5]:
merged_df = pd.DataFrame(llm_result.get("merged_senses", []))
merged_df

,sense_id,canonical_label,merged_description,source_candidate_ids,merge_rationale
0,s1,time of year,A recurring subdivision of the year marked by ...,"[1, 2]",Both candidates describe a general year subdiv...
1,s2,sports season,The scheduled period during which a sports lea...,[3],This is a distinct sports-specific sense with ...
2,s3,TV season,A set of episodes produced as one installment ...,[4],This is a distinct media sense referring to a ...


In [6]:
discarded_df = pd.DataFrame(llm_result.get("discarded_candidates", []))
discarded_df

""


## 4. Save Experiment Output

In [7]:
# OUTPUT_DIR = REPO_ROOT / "cache" / "wikidata_llm_candidate_merge"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# output_path = OUTPUT_DIR / f"{WORD.replace(' ', '_')}_max{MAX_CANDIDATE_COUNT}_merged.json"
#
# record = {
#     "word": WORD,
#     "max_candidate_count": MAX_CANDIDATE_COUNT,
#     "use_detailed_description": USE_DETAILED_DESCRIPTION,
#     "openai_model": OPENAI_MODEL,
#     "token_usage": token_usage,
#     "raw_candidates": candidate_senses,
#     "llm_result": llm_result,
# }
# output_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
# print(output_path)